In [13]:
import pandas as pd

In [14]:
#loading and orienting the data
expr_raw = pd.read_csv('brca_metabric/data_mrna_illumina_microarray.txt', sep='\t', comment='#')

#drop rows with no gene symbol, collapse duplicate symbols by mean
expr_raw = expr_raw.dropna(subset=['Hugo_Symbol'])
expr_raw = expr_raw.drop(columns=['Entrez_Gene_Id'])
expr_raw = expr_raw.groupby('Hugo_Symbol').mean(numeric_only=True)
expr_raw.head()

,MB-0362,MB-0346,MB-0386,MB-0574,MB-0185,MB-0503,MB-0641,MB-0201,MB-0218,MB-0316,...,MB-6192,MB-4820,MB-5527,MB-5167,MB-5465,MB-5453,MB-5471,MB-5127,MB-4313,MB-4823
Hugo_Symbol,,,,,,,,,,,,,,,,,,,,,
A1BG,5.662294,5.854950,5.743328,5.716885,5.504649,5.948696,5.725525,5.734731,5.788228,5.763666,...,5.501222,5.649023,5.654469,5.642720,5.431803,5.732303,5.503984,5.550886,6.101505,5.633491
A1CF,5.350444,5.609458,5.435237,5.571744,5.257009,5.233115,5.311067,5.273068,5.207229,5.347844,...,5.613889,5.317775,5.652508,5.507495,5.202999,5.365232,5.285782,5.488379,5.356509,5.689814
A2M,7.653319,6.564312,8.201633,7.587296,8.427318,7.545219,8.693606,8.898809,8.763382,9.100264,...,7.736627,9.005442,8.696260,8.796967,10.127953,11.078945,8.714076,8.333257,10.018235,9.466299
A2ML1,5.535317,5.938013,5.337279,5.221707,5.514548,5.279701,5.201008,5.695971,5.108838,6.942946,...,5.265792,5.254453,7.689091,5.363285,5.865814,5.378435,5.600398,5.247741,5.421864,6.040639
A4GALT,6.347507,5.567505,6.923349,5.756924,6.154819,6.407954,7.228863,6.608221,5.947872,6.951467,...,7.053883,5.849546,6.460017,7.708127,7.016025,6.598701,6.837395,6.093857,7.225162,6.353837


In [15]:
#transpose: rows = patients, columns = genes
expr = expr_raw.transpose()
expr.index.name = 'PATIENT_ID'
expr = expr.reset_index()

print(expr.shape)
expr.head()

(1980, 20386)


Hugo_Symbol,PATIENT_ID,A1BG,A1CF,A2M,A2ML1,A4GALT,A4GNT,AA06,AAA1,AAAS,...,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,hCG_1997999,hCG_2023280,psiTPTE22
0,MB-0362,5.662294,5.350444,7.653319,5.535317,6.347507,5.264170,5.259617,5.391005,6.479471,...,5.845263,6.434705,5.321824,8.674482,9.018941,7.558310,7.003461,6.008556,5.355926,5.269515
1,MB-0346,5.854950,5.609458,6.564312,5.938013,5.567505,5.285960,5.179630,5.377596,6.511604,...,6.191387,6.567050,5.388879,8.091321,8.292709,7.159059,7.354656,5.772867,5.350325,5.012334
2,MB-0386,5.743328,5.435237,8.201633,5.337279,6.923349,5.275596,5.467693,5.127266,6.183451,...,5.690757,6.381255,5.429683,8.759179,9.162176,7.484611,6.853452,5.621497,5.428751,5.401948
3,MB-0574,5.716885,5.571744,7.587296,5.221707,5.756924,5.209861,5.222671,5.259309,6.079376,...,6.612426,6.919791,5.093761,8.778374,8.169537,7.639262,7.240531,5.509456,5.263631,5.227373
4,MB-0185,5.504649,5.257009,8.427318,5.514548,6.154819,5.613277,5.497661,5.443880,6.401032,...,5.982131,6.363514,5.329793,8.481611,8.787354,7.447219,7.118362,5.567187,5.329880,5.250153


In [16]:
#QC: check for missing values
#.drop() vs .dropna() - drop() removes columns or rows by name, dropna() removes rows or columns with missing values
missing_per_gene = expr.drop(columns='PATIENT_ID').isnull().mean() #this is to find the proportion of missing values per gene 
genes_to_drop = missing_per_gene[missing_per_gene > 0.2].index #drop genes with more than 20% missing values
expr = expr.drop(columns=genes_to_drop) 
print(f"Dropped {len(genes_to_drop)} genes with >20% missing values. Remaining genes: {expr.shape[1]-1}") #shape[1]-1 because we don't want to count the PATIENT_ID column

Dropped 0 genes with >20% missing values. Remaining genes: 20385


In [ ]:
#Fill any remaining missing values with the gene-wise median (safer than mean because it is less sensitive to outliers, for expression data)
gene_cols = expr.columns.drop('PATIENT_ID')
expr[gene_cols] = expr[gene_cols].fillna(expr[gene_cols].median())
expr.head()

Hugo_Symbol,PATIENT_ID,A1BG,A1CF,A2M,A2ML1,A4GALT,A4GNT,AA06,AAA1,AAAS,...,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,hCG_1997999,hCG_2023280,psiTPTE22
0,MB-0362,5.662294,5.350444,7.653319,5.535317,6.347507,5.264170,5.259617,5.391005,6.479471,...,5.845263,6.434705,5.321824,8.674482,9.018941,7.558310,7.003461,6.008556,5.355926,5.269515
1,MB-0346,5.854950,5.609458,6.564312,5.938013,5.567505,5.285960,5.179630,5.377596,6.511604,...,6.191387,6.567050,5.388879,8.091321,8.292709,7.159059,7.354656,5.772867,5.350325,5.012334
2,MB-0386,5.743328,5.435237,8.201633,5.337279,6.923349,5.275596,5.467693,5.127266,6.183451,...,5.690757,6.381255,5.429683,8.759179,9.162176,7.484611,6.853452,5.621497,5.428751,5.401948
3,MB-0574,5.716885,5.571744,7.587296,5.221707,5.756924,5.209861,5.222671,5.259309,6.079376,...,6.612426,6.919791,5.093761,8.778374,8.169537,7.639262,7.240531,5.509456,5.263631,5.227373
4,MB-0185,5.504649,5.257009,8.427318,5.514548,6.154819,5.613277,5.497661,5.443880,6.401032,...,5.982131,6.363514,5.329793,8.481611,8.787354,7.447219,7.118362,5.567187,5.329880,5.250153


In [18]:
#Cell: load both datasets
clinical_final = pd.read_csv('brca_metabric/clinical_final.csv')

#Cell: diagnostic check
missing_subtype_ids = clinical_final.loc[clinical_final['CLAUDIN_SUBTYPE'].isnull(), 'PATIENT_ID']
overlap = expr['PATIENT_ID'].isin(missing_subtype_ids).sum()

if len(missing_subtype_ids) == 0:
    print("No patients are missing CLAUDIN_SUBTYPE.")
else:
    print(f"Of {len(missing_subtype_ids)} patients missing CLAUDIN_SUBTYPE, {overlap} have expression data anyway.")


#Cell: build merged DEG-analysis dataframe
merged = expr.merge(clinical_final[['PATIENT_ID','CLAUDIN_SUBTYPE']], on='PATIENT_ID')
merged = merged.dropna(subset=['CLAUDIN_SUBTYPE'])
merged = merged[merged['CLAUDIN_SUBTYPE'] != 'NC']
print(merged['CLAUDIN_SUBTYPE'].value_counts())

Of 529 patients missing CLAUDIN_SUBTYPE, 0 have expression data anyway.
CLAUDIN_SUBTYPE
LumA           700
LumB           475
Her2           224
claudin-low    218
Basal          209
Normal         148
Name: count, dtype: int64


In [ ]:
#Cell: check for patient ID overlap between gene expression and clinical data
clinical = pd.read_csv('brca_metabric/clinical_final.csv')
expr_ids = set(expr['PATIENT_ID'])
clin_ids = set(clinical['PATIENT_ID'])

print("expr count:", len(expr_ids))
print("clinical count:", len(clin_ids))
print("intersection:", len(expr_ids & clin_ids))
print("in clinical not expr (sample):", list(clin_ids - expr_ids)[:10])
print("in expr not clinical (sample):", list(expr_ids - clin_ids)[:10])

expr count: 1980
clinical count: 2509
intersection: 1980
in clinical not expr (sample): ['MTS-T2304', 'MTS-T0387', 'MTS-T0378', 'MTS-T1272', 'MB-0025', 'MTS-T0332', 'MTS-T2423', 'MTS-T0266', 'MTS-T1280', 'MTS-T1633']
in expr not clinical (sample): []


In [22]:
gene_cols = expr.columns.drop('PATIENT_ID')
print(expr[gene_cols].dtypes.value_counts())
print("Duplicate gene columns:", expr.columns.duplicated().sum())

float64    20385
Name: count, dtype: int64
Duplicate gene columns: 0


In [ ]:
#Cell: DEG analysis is performed using ANOVA F-test to find genes that are differentially expressed across the CLAUDIN_SUBTYPE groups. The top 500 genes with the smallest p-values are selected as DEGs.
from sklearn.feature_selection import f_classif
import pandas as pd

gene_cols = expr.columns.drop('PATIENT_ID')

# ensure numeric dtype (fixes it if diagnostic above showed object dtype)
X_genes = merged[gene_cols].apply(pd.to_numeric, errors='coerce')
y_labels = merged['CLAUDIN_SUBTYPE'].values

f_scores, p_values = f_classif(X_genes.values, y_labels)

deg_pvals = pd.Series(p_values, index=gene_cols).sort_values()
top_genes = deg_pvals.head(500).index.tolist()

print(f"Selected {len(top_genes)} DEGs, smallest p-value: {deg_pvals.min():.2e}")
deg_pvals.to_csv('brca_metabric/deg_pvalues.csv')

Selected 500 DEGs, smallest p-value: 0.00e+00


In [26]:
print(deg_pvals.head(15))

Hugo_Symbol
ESR1         0.000000e+00
FOXC1        0.000000e+00
FOXA1        0.000000e+00
AGR3         0.000000e+00
PSAT1        0.000000e+00
MLPH         0.000000e+00
CA12         0.000000e+00
TBC1D9       0.000000e+00
GATA3        0.000000e+00
BCL11A      3.148087e-318
SPDEF       4.079149e-316
XBP1        8.451330e-304
FAM171A1    6.780699e-300
TFF3        2.400883e-285
HAPLN3      3.838800e-282
dtype: float64
